In [1]:
import caf.base as cb
import caf.tem as ct
import pandas as pd
import os
from pathlib import Path

# HB Production
## Preprocessing
### Create equivalent population DVector

Nhan's code writes a .csv called pop_2023.csv\
It comes from landuse Output P11, the code translates and adds aws to the segmentation, amongst other actions.

Create DVector from Nhan's Output P11 derived pop_2023.csv...\
zoning: lsoa_2021\
segmentation: [gender_3, aws, ns_sec, soc, accom_hh]

In [2]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\lu_pop_2023.hdf"
if not os.path.exists(dvec_path):
    pop = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\lu_pop_2023.csv")
    pop = pop.set_index(["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"])#.drop(columns=["adult_nssec"])
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"],
                                            naming_order=["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"]))
    zoning_system = cb.ZoningSystem.get_zoning("lsoa_2021")
    pop_dvec = cb.DVector(segmentation=segmentation, import_data=pop, zoning_system=zoning_system)
    pop_dvec.save(dvec_path)

### Create equivalent trip rates DVector

In [3]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\hb_trip_rates_production.hdf"
if not os.path.exists(dvec_path):
    tr = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\hb_trip_rates_production.csv")
    tr = tr.rename(columns={"gender": "gender_3", "ns": "ns_sec", "purpose": "p"}).pivot(index=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"], columns="tfn_at", values="beta")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"],
                                                        naming_order=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"]))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    tr_dvec = cb.DVector(segmentation=segmentation, import_data=tr, zoning_system=zoning_system)
    tr_dvec.save(dvec_path)

### Create equivalent 2023 adjustment DVector

In [4]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\trip_rate_adjustments_production_hb_fr.hdf"
if not os.path.exists(dvec_path):
    adj = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\trip_rate_adjustments.csv")
    adj = adj.rename(columns={"purpose": "p"}).loc[(adj["pa"]=="p") & (adj["direction"]=="hb_fr")].pivot(index=["p"], columns="gor", values="adj")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p"],
                                                        naming_order=["p"]))
    zoning_system = cb.ZoningSystem.get_zoning("gor")
    adj_dvec = cb.DVector(segmentation=segmentation, import_data=adj, zoning_system=zoning_system)
    adj_dvec.save(dvec_path)

### Create equivalent mts DVector

In [5]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\mode_time_split_production_hb_fr_reg.hdf"
if not os.path.exists(dvec_path):
    mts = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\mode_time_split_production_hb_fr_reg.csv")
    mts = mts.rename(columns={"mode": "m", "period": "tp", "purpose": "p"}).pivot(index=["p", "m", "tp", "hh_type"], columns="tfn_at", values="rho")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "m", "tp", "hh_type"],
                                                        naming_order=["p", "m", "tp", "hh_type"]))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    mts_dvec = cb.DVector(segmentation=segmentation, import_data=mts, zoning_system=zoning_system)
    mts_dvec.save(dvec_path)

## TEM Setup

In [6]:
tem = ct.TEM(
    model_years=[2023],
    scenario="Core",
    output_zoning="normits",
    iteration_name="20250304",
    export_home=r"T:\ThomasPrince\TEM Input\comparison\caf.tem output",
    return_segmentation=["hh_type", "p", "m", "tp"]
)

In [7]:
input_dir = Path(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input")

HBProd = tem.HBProductionModel(
    population_paths={2023: input_dir / "lu_pop_2023.hdf"},
    trip_rates_path=input_dir / "hb_trip_rates_production.hdf",
    mode_time_splits_path=input_dir / "mode_time_split_production_hb_fr_reg.hdf",
    adjustment_path=input_dir / "trip_rate_adjustments_production_hb_fr.hdf",
    population_translation_path=r"T:\ThomasPrince\TEM Input\lsoa_normits_pop.csv"
)

In [8]:
HBProd.run()

C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\caf.base\src\caf\base\segmentation.py:342: SegmentationWarning: Read in level p is a subset of the segment. If this was not expected check the input segmentation.
  warnings.warn(
C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\caf.base\src\caf\base\zoning.py:179: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['1001001' '1001002' '1001003' ... '11358007' '11358008' '11358009']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  zones.loc[:, name] = zones[name].astype(str)
C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\caf.base\src\caf\base\zoning.py:681: TranslationWarning: 20 tfn_at zones have splitting factors which don't sum to 1 (value totals may change during zone_translation), the maximum difference is 7.7e+02
  warnings.warn(
C:\Users\Spiral\Documents\Thomas Princ

In [ ]:
cb.DVector.load(HBProd.model.export_paths.pure_demand[2023]).aggregate(["p"]).data

In [9]:
check = cb.DVector.load(HBProd.model.export_paths.pure_demand[2023])#.translate_zoning(cb.ZoningSystem.get_zoning("gor"))

In [ ]:
check.aggregate(["p"]).data.rename(columns=check.zoning_system.id_to_desc)

In [10]:
pop_2023 = pd.read_csv(r"C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\NTS-Processing_python-refactor\NoTEM\voa_gb_2023_uni\reports\pop_2023_normits.csv")
pop_2023 = pop_2023.groupby(["normits_v3.3_id"])[["1","2","3","4","5","6","7","8"]].sum().T
pop_2023.index = pop_2023.index.astype(int)

In [15]:
test = (check.aggregate(["p"]).data - pop_2023).stack()
test = test.reset_index()
test = test.groupby("normits_id")[0].sum()
test.loc[abs(test)>1]#.to_csv(r"C:\Users\Spiral\Documents\Thomas Prince\tem_minus_ntsprocessing.csv")

normits_id
5248009    2715.089252
Name: 0, dtype: float64

In [ ]:
check.aggregate(["p"]).data.sum().sum() / pop_2023.sum().sum()

In [ ]:
pd.DataFrame(test.loc[abs(test)>10].index.groupby("normits_id").sum())#.to_csv()

In [ ]:
pop_2023

In [ ]:
check.aggregate(["p"]).data.columns

In [ ]:
test#.sum().sum()

In [ ]:
check.aggregate(["p"]).data.sum().sum()

In [ ]:
check.aggregate(["p"]).data.sum().sum()/pop_2023.sum().sum()

In [ ]:
324*8

In [29]:
test = pd.read_csv(r"C:\Users\Spiral\Documents\Thomas Prince\test1.csv")

In [35]:
test = pd.read_csv(r"C:\Users\Spiral\Documents\Thomas Prince\test1.csv")
test = test.groupby(["gor", "purpose"])["trips"].sum().reset_index()
test = test.pivot(index = ["purpose"], columns = "gor", values="trips")

In [ ]:
test

In [ ]:
336/8

In [ ]:
check.zoning_system.id_to_desc